In [ ]:
%pip install -q --upgrade sentence-transformers chromadb transformers accelerate


In [ ]:
print("If packages were just installed/updated, re-run this notebook from Cell 2 onward.")


In [ ]:
import os
import re
import heapq
from datetime import datetime
from typing import Dict, List

import numpy as np
import chromadb
from chromadb.config import Settings
from transformers import pipeline
from sentence_transformers import SentenceTransformer


os.environ["ANONYMIZED_TELEMETRY"] = "False"


def log(msg: str):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")


In [ ]:
EMBEDDING_DELTA_PATH = "/Volumes/workspace/legal_data/vector_db_test/legal_embeddings_delta"
CHROMA_DB_CANDIDATES = [
    "/Volumes/workspace/legal_data/chroma_db/legal_knowledge_test",
    "/Volumes/workspace/legal_data/vector_db_test/chroma_db_legal_knowledge_test",
]
COLLECTION_NAME = "legal_knowledge"

PRIMARY_EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
FALLBACK_EMBED_MODEL = "sentence-transformers/paraphrase-MiniLM-L3-v2"

PRIMARY_QA_MODEL = "google/flan-t5-base"   # more stable on serverless than large
FALLBACK_QA_MODEL = "google/flan-t5-small"

TOP_K = 8


In [ ]:
def load_embedding_model():
    for model_name in [PRIMARY_EMBED_MODEL, FALLBACK_EMBED_MODEL]:
        try:
            log(f"Loading embedding model: {model_name}")
            model = SentenceTransformer(model_name)
            _ = model.encode(["health check"], show_progress_bar=False)
            log(f"Embedding model ready: {model_name}")
            return model, model_name
        except Exception as e:
            log(f"Embedding model failed ({model_name}): {e}")
    return None, None


def load_qa_pipeline():
    for model_name in [PRIMARY_QA_MODEL, FALLBACK_QA_MODEL]:
        try:
            log(f"Loading QA model: {model_name}")
            qa = pipeline(
                "text2text-generation",
                model=model_name,
                max_new_tokens=256,
                do_sample=False,
                temperature=0.0,
            )
            log(f"QA model ready: {model_name}")
            return qa, model_name
        except Exception as e:
            log(f"QA model failed ({model_name}): {e}")
    return None, None


def load_chroma_collection():
    errors = []
    for candidate in CHROMA_DB_CANDIDATES:
        try:
            os.makedirs(candidate, exist_ok=True)
            client = chromadb.PersistentClient(
                path=candidate,
                settings=Settings(anonymized_telemetry=False, allow_reset=True),
            )
            collection = client.get_or_create_collection(COLLECTION_NAME)
            return collection, candidate, None
        except Exception as e:
            errors.append(f"{candidate}: {e}")

    return None, None, " | ".join(errors)


def load_embedding_delta():
    try:
        df = spark.read.format("delta").load(EMBEDDING_DELTA_PATH)
        needed = ["chunk_id", "chunk_text", "act_name", "section_number", "category", "file_name", "embedding"]
        missing = [c for c in needed if c not in df.columns]
        if missing:
            return None, f"Embedding Delta missing columns: {missing}"

        clean_df = (
            df.select(*needed)
              .dropna(subset=["chunk_id", "chunk_text", "embedding"])
              .dropDuplicates(["chunk_id"])
        )

        cnt = clean_df.count()
        if cnt == 0:
            return None, "Embedding Delta exists but has 0 rows"

        return clean_df, None
    except Exception as e:
        return None, str(e)


def hydrate_chroma_from_delta(collection, embeddings_df, batch_size=200):
    if collection is None or embeddings_df is None:
        return 0

    inserted = 0
    rows = []

    def flush(batch_rows):
        if not batch_rows:
            return 0

        ids = []
        docs = []
        embeds = []
        metas = []

        for r in batch_rows:
            if not r.chunk_id or not r.chunk_text or not r.embedding:
                continue
            ids.append(str(r.chunk_id))
            docs.append(str(r.chunk_text))
            embeds.append([float(x) for x in r.embedding])
            metas.append({
                "act_name": str(r.act_name or ""),
                "section": str(r.section_number or ""),
                "category": str(r.category or ""),
                "source": str(r.file_name or ""),
            })

        if not ids:
            return 0

        collection.upsert(ids=ids, documents=docs, embeddings=embeds, metadatas=metas)
        return len(ids)

    for row in embeddings_df.toLocalIterator():
        rows.append(row)
        if len(rows) >= batch_size:
            inserted += flush(rows)
            rows = []

    if rows:
        inserted += flush(rows)

    return inserted


In [ ]:
embedding_model, embedding_model_name = load_embedding_model()
qa_model, qa_model_name = load_qa_pipeline()
collection, chroma_path, chroma_error = load_chroma_collection()
embeddings_df, delta_error = load_embedding_delta()

if chroma_error:
    log(f"Chroma init warning: {chroma_error}")
else:
    log(f"Chroma collection path: {chroma_path}")
    log(f"Chroma collection count before hydration: {collection.count()}")

if delta_error:
    log(f"Embedding Delta warning: {delta_error}")
else:
    log(f"Embedding Delta rows available: {embeddings_df.count()}")

# Auto-heal: if Chroma is empty but Delta exists, hydrate Chroma from Delta
if collection is not None and embeddings_df is not None:
    try:
        current = collection.count()
        if current == 0:
            inserted = hydrate_chroma_from_delta(collection, embeddings_df)
            log(f"Hydrated Chroma from Delta. Inserted/updated vectors: {inserted}")
            log(f"Chroma collection count after hydration: {collection.count()}")
    except Exception as e:
        log(f"Chroma hydration warning: {e}")

print("--- Runtime Status ---")
print("Embedding model:", embedding_model_name or "Unavailable")
print("QA model:", qa_model_name or "Unavailable")
print("Chroma available:", collection is not None)
print("Delta available:", embeddings_df is not None)


In [ ]:
def embed_query(query: str):
    if embedding_model is None:
        return None
    try:
        return embedding_model.encode([query], show_progress_bar=False)[0]
    except Exception as e:
        log(f"Query embedding failed: {e}")
        return None


def retrieve_from_chroma(query: str, k: int = 5):
    if collection is None:
        return [], []

    q = embed_query(query)
    if q is None:
        return [], []

    try:
        res = collection.query(
            query_embeddings=[q.tolist()],
            n_results=k,
        )
        docs = res.get("documents", [[]])[0]
        metas = res.get("metadatas", [[]])[0]
        return docs, metas
    except Exception as e:
        log(f"Chroma retrieval failed: {e}")
        return [], []


def retrieve_from_delta_cosine(query: str, k: int = 5):
    if embeddings_df is None:
        return [], []

    q = embed_query(query)
    if q is None:
        return [], []

    q = np.array(q, dtype=np.float32)
    q_norm = float(np.linalg.norm(q)) + 1e-12

    heap = []
    seq = 0

    cols = ["chunk_text", "act_name", "section_number", "category", "file_name", "embedding"]
    for row in embeddings_df.select(*cols).toLocalIterator():
        emb = row.embedding
        if not emb:
            continue

        v = np.array(emb, dtype=np.float32)
        denom = (float(np.linalg.norm(v)) + 1e-12) * q_norm
        score = float(np.dot(q, v) / denom)

        meta = {
            "act_name": str(row.act_name or ""),
            "section": str(row.section_number or ""),
            "category": str(row.category or ""),
            "source": str(row.file_name or ""),
        }

        item = (score, seq, str(row.chunk_text), meta)
        seq += 1

        if len(heap) < k:
            heapq.heappush(heap, item)
        else:
            heapq.heappushpop(heap, item)

    top = sorted(heap, key=lambda x: x[0], reverse=True)
    docs = [x[2] for x in top]
    metas = [x[3] for x in top]
    return docs, metas


def retrieve_from_delta_keyword(query: str, k: int = 5):
    if embeddings_df is None:
        return [], []

    terms = [t for t in re.findall(r"[a-zA-Z0-9]+", query.lower()) if len(t) > 2]
    if not terms:
        return [], []

    scored = []
    for row in embeddings_df.select("chunk_text", "act_name", "section_number", "category", "file_name").toLocalIterator():
        text = str(row.chunk_text or "")
        if not text:
            continue

        low = text.lower()
        score = sum(1 for t in terms if t in low)
        if score == 0:
            continue

        meta = {
            "act_name": str(row.act_name or ""),
            "section": str(row.section_number or ""),
            "category": str(row.category or ""),
            "source": str(row.file_name or ""),
        }

        scored.append((score, text, meta))

    scored.sort(key=lambda x: x[0], reverse=True)
    top = scored[:k]
    docs = [x[1] for x in top]
    metas = [x[2] for x in top]
    return docs, metas


def retrieve_chunks(query: str, k: int = TOP_K):
    # priority: Chroma -> Delta cosine -> Delta keyword
    docs, metas = retrieve_from_chroma(query, k)
    if docs:
        return docs, metas, "chroma"

    docs, metas = retrieve_from_delta_cosine(query, k)
    if docs:
        return docs, metas, "delta_cosine"

    docs, metas = retrieve_from_delta_keyword(query, k)
    if docs:
        return docs, metas, "delta_keyword"

    return [], [], "none"


In [ ]:
def refine_context(docs: List[str], max_docs: int = 4) -> List[str]:
    seen = set()
    refined = []

    for doc in docs:
        snippet = (doc or "").strip()
        if not snippet:
            continue

        # keep smaller legal snippets too; avoid over-filtering
        if snippet not in seen:
            refined.append(snippet)
            seen.add(snippet)

        if len(refined) >= max_docs:
            break

    return refined


In [ ]:
def build_prompt(query: str, context: List[str]) -> str:
    context_text = "\n\n".join(context) if context else "No context available."

    return f"""
You are a legal assistant helping Indian citizens understand laws.

Use the legal context below to answer clearly and factually.
- Use simple language.
- Mention legal section numbers if present.
- Mention penalties/fines/punishment if present.
- Give practical citizen guidance.
- If context is insufficient, say that clearly.
- Do not repeat the prompt.

Question:
{query}

Legal Context:
{context_text}

Answer:
"""


def generate_fallback_answer(query: str, context: List[str]) -> str:
    if not context:
        return (
            "I could not find relevant legal context in the current index. "
            "Please ensure embeddings are generated from legal chunks and rerun retrieval."
        )

    joined = " ".join(context)
    short = joined[:900]
    return (
        "Based on retrieved legal text, here is the relevant information:\n\n"
        f"{short}\n\n"
        "This is a context-based summary because the generation model is unavailable in this session."
    )


def generate_answer(query: str):
    docs, metadata, source = retrieve_chunks(query, TOP_K)
    context = refine_context(docs)

    if not context:
        return "No relevant legal context found in the vector database.", metadata, source

    if qa_model is None:
        return generate_fallback_answer(query, context), metadata, source

    prompt = build_prompt(query, context)

    try:
        response = qa_model(prompt)[0]["generated_text"].strip()
        if not response:
            response = generate_fallback_answer(query, context)
        return response, metadata, source
    except Exception as e:
        log(f"Generation warning: {e}")
        return generate_fallback_answer(query, context), metadata, source


In [ ]:
def format_output(answer: str, metadata: List[Dict], retrieval_source: str) -> str:
    sections = sorted({m.get("section", "") for m in metadata if m.get("section")})

    return f"""
LEGAL EXPLANATION:

{answer}

Relevant Sections:
{", ".join(sections) if sections else "Refer to applicable legal provisions"}

Retrieval Source:
{retrieval_source}

Disclaimer:
This response is AI-generated legal information and not a substitute for professional legal advice.
"""


In [ ]:
query = "What is the penalty for not wearing a helmet in India?"

answer, metadata, retrieval_source = generate_answer(query)
print(format_output(answer, metadata, retrieval_source))


In [ ]:
# Run additional tests
for q in [
    "What is the penalty for not wearing a helmet?",
    "What does Section 129 of Motor Vehicles Act say?",
    "Can triple riding on a bike lead to fine?",
]:
    print("\n" + "=" * 100)
    print("Query:", q)
    ans, meta, src = generate_answer(q)
    print(format_output(ans, meta, src))
